In [37]:
import os
import glob
import re
import numpy as np
from scipy.stats import ttest_rel, wilcoxon
import cv2
from sklearn.metrics import jaccard_score
from PIL import Image

# For UNet

In [38]:
def load_image(filepath):
    # Load the image using PIL
    image = np.array(Image.open(filepath).convert('L'))
    
    # Apply Otsu's thresholding
    t, binary_image = cv2.threshold(image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    # Convert to binary (0 and 1)
    binary_image = (binary_image > 0).astype(np.uint8)
    return binary_image

def dice_coefficient(y_true, y_pred, smooth=1):
    y_true_f = y_true.flatten()
    y_pred_f = y_pred.flatten()
    intersection = np.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (np.sum(y_true_f) + np.sum(y_pred_f) + smooth)

# -----------------------------
# Paths
# -----------------------------
corruptions = [2,12,15,17,20,25,30]

for corruption in corruptions:
    print(f"Corruption: {corruption}")
    gt_folder = './GEE_Masks/GEE_resized/test_gee/'
    modelA_folder = f'./GEE_Output/UNet/0'      # vanilla
    modelB_folder = f'./GEE_Output/UNet/{corruption}'     # corrupted (example)

    # -----------------------------
    # Helper: extract numeric index
    # -----------------------------
    def extract_index(filename):
        match = re.search(r'(\d+)', os.path.basename(filename))
        return int(match.group(1)) if match else -1

    # -----------------------------
    # Collect prediction files
    # -----------------------------
    pattern = re.compile(r'^dense_\d+\.tif$')

    predA_files = sorted(
        [
            f for f in glob.glob(os.path.join(modelA_folder, '*.tif'))
            if pattern.match(os.path.basename(f))
        ],
        key=extract_index
    )

    predB_files = sorted(
        [
            f for f in glob.glob(os.path.join(modelB_folder, '*.tif'))
            if pattern.match(os.path.basename(f))
        ],
        key=extract_index
    )

    assert len(predA_files) == len(predB_files), "Mismatch in number of test samples"

    # -----------------------------
    # Metric storage
    # -----------------------------
    dice_A, dice_B = [], []
    iou_A, iou_B = [], []

    # -----------------------------
    # Evaluation loop
    # -----------------------------
    for fileA, fileB in zip(predA_files, predB_files):

        idxA = extract_index(fileA)
        idxB = extract_index(fileB)
        assert idxA == idxB, "Prediction files are misaligned"

        gt_path = os.path.join(gt_folder, f'NDWI_Mask_{idxA}_resized.tif')
        if not os.path.exists(gt_path):
            continue

        # Load images (assumed binary already)
        gt = load_image(gt_path).flatten()
        predA = load_image(fileA).flatten()
        predB = load_image(fileB).flatten()

        # Metrics
        dice_A.append(dice_coefficient(gt, predA))
        dice_B.append(dice_coefficient(gt, predB))

        iou_A.append(jaccard_score(gt, predA, average='macro'))
        iou_B.append(jaccard_score(gt, predB, average='macro'))

    # Convert to numpy arrays
    dice_A = np.array(dice_A)
    dice_B = np.array(dice_B)
    iou_A  = np.array(iou_A)
    iou_B  = np.array(iou_B)

    dice_diff = dice_B - dice_A
    iou_diff  = iou_B  - iou_A

    t_dice, p_dice = ttest_rel(dice_A, dice_B)
    t_iou,  p_iou  = ttest_rel(iou_A, iou_B)

    w_dice, p_w_dice = wilcoxon(dice_diff, zero_method='wilcox')
    w_iou,  p_w_iou  = wilcoxon(iou_diff,  zero_method='wilcox')

    # -----------------------------
    # Report
    # -----------------------------
    print(f"Dice (Model B − Model A): mean diff = {np.mean(dice_diff):.4f}")
    print(f"Paired t-test Dice:     t = {t_dice:.4f}, p = {p_dice:.4f}")
    print(f"Wilcoxon test Dice:     W = {w_dice:.4f}, p = {p_w_dice:.4f}")
    print(f"IoU  (Model B − Model A): mean diff = {np.mean(iou_diff):.4f}")
    print(f"Paired t-test IoU:      t = {t_iou:.4f},  p = {p_iou:.4f}")
    print(f"Wilcoxon test IoU:      W = {w_iou:.4f},  p = {p_w_iou:.4f}")
    print()


Corruption: 2
Dice (Model B − Model A): mean diff = 0.0052
Paired t-test Dice:     t = -0.8617, p = 0.3897
Wilcoxon test Dice:     W = 8681.0000, p = 0.0000
IoU  (Model B − Model A): mean diff = 0.0022
Paired t-test IoU:      t = -0.5532,  p = 0.5806
Wilcoxon test IoU:      W = 7130.0000,  p = 0.0000

Corruption: 12
Dice (Model B − Model A): mean diff = -0.0003
Paired t-test Dice:     t = 0.0448, p = 0.9643
Wilcoxon test Dice:     W = 6756.5000, p = 0.0000
IoU  (Model B − Model A): mean diff = -0.0006
Paired t-test IoU:      t = 0.1414,  p = 0.8877
Wilcoxon test IoU:      W = 7743.5000,  p = 0.0000

Corruption: 15
Dice (Model B − Model A): mean diff = -0.0356
Paired t-test Dice:     t = 4.9790, p = 0.0000
Wilcoxon test Dice:     W = 1653.0000, p = 0.0000
IoU  (Model B − Model A): mean diff = -0.0274
Paired t-test IoU:      t = 6.2284,  p = 0.0000
Wilcoxon test IoU:      W = 1920.0000,  p = 0.0000

Corruption: 17
Dice (Model B − Model A): mean diff = -0.0116
Paired t-test Dice:     t = 

# For other models

In [39]:

# def load_image(filepath):
#     # Load the image using PIL
#     image = np.array(Image.open(filepath).convert('L'))
    
#     # Apply Otsu's thresholding
#     t, binary_image = cv2.threshold(image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
#     # Convert to binary (0 and 1)
#     binary_image = (binary_image > 0).astype(np.uint8)
#     return binary_image

# def dice_coefficient(y_true, y_pred, smooth=1):
#     y_true_f = y_true.flatten()
#     y_pred_f = y_pred.flatten()
#     intersection = np.sum(y_true_f * y_pred_f)
#     return (2. * intersection + smooth) / (np.sum(y_true_f) + np.sum(y_pred_f) + smooth)

# # -----------------------------
# # Paths
# # -----------------------------
# corruptions = [2,12,15,17,20,25,30]
# model = "DeepLabOutputs"

# for corruption in corruptions:
    
#     print(f"Corruption: {corruption}")
    
#     gt_folder = './sar_images/masks/test'
#     modelA_folder = f'./GEE_Output/{model}/0/Outputs'      # vanilla
#     modelB_folder = f'./GEE_Output/{model}/{corruption}/Outputs'     # corrupted (example)

#     # -----------------------------
#     # Helper: extract numeric index
#     # -----------------------------
#     def extract_index(filename):
#         match = re.search(r'(\d+)', os.path.basename(filename))
#         return int(match.group(1)) if match else -1

#     # -----------------------------
#     # Collect prediction files
#     # -----------------------------
#     predA_files = sorted(
#         glob.glob(os.path.join(modelA_folder, '*.png')),
#         key=extract_index
#     )

#     predB_files = sorted(
#         glob.glob(os.path.join(modelB_folder, '*.png')),
#         key=extract_index
#     )

#     assert len(predA_files) == len(predB_files), "Mismatch in number of test samples"

#     # -----------------------------
#     # Metric storage
#     # -----------------------------
#     dice_A, dice_B = [], []
#     iou_A, iou_B = [], []

#     # -----------------------------
#     # Evaluation loop
#     # -----------------------------
#     for fileA, fileB in zip(predA_files, predB_files):

#         idxA = extract_index(fileA)
#         idxB = extract_index(fileB)
#         assert idxA == idxB, "Prediction files are misaligned"

#         gt_path = os.path.join(gt_folder, f'{idxA}.png')
#         if not os.path.exists(gt_path):
#             continue

#         # Load images (assumed binary already)
#         gt = load_image(gt_path).flatten()
#         predA = load_image(fileA).flatten()
#         predB = load_image(fileB).flatten()

#         # Metrics
#         dice_A.append(dice_coefficient(gt, predA))
#         dice_B.append(dice_coefficient(gt, predB))

#         iou_A.append(jaccard_score(gt, predA, average='macro'))
#         iou_B.append(jaccard_score(gt, predB, average='macro'))

#     # Convert to numpy arrays
#     dice_A = np.array(dice_A)
#     dice_B = np.array(dice_B)
#     iou_A  = np.array(iou_A)
#     iou_B  = np.array(iou_B)

#     # -----------------------------
#     # Paired t-tests
#     # -----------------------------
#     dice_diff = dice_B - dice_A
#     iou_diff  = iou_B  - iou_A

#     t_dice, p_dice = ttest_rel(dice_A, dice_B)
#     t_iou,  p_iou  = ttest_rel(iou_A, iou_B)

#     w_dice, p_w_dice = wilcoxon(dice_diff, zero_method='wilcox')
#     w_iou,  p_w_iou  = wilcoxon(iou_diff,  zero_method='wilcox')

#     # -----------------------------
#     # Report
#     # -----------------------------
#     print(f"Dice (Model B − Model A): mean diff = {np.mean(dice_diff):.4f}")
#     print(f"Paired t-test Dice:     t = {t_dice:.4f}, p = {p_dice:.4f}")
#     print(f"Wilcoxon test Dice:     W = {w_dice:.4f}, p = {p_w_dice:.4f}")
#     print(f"IoU  (Model B − Model A): mean diff = {np.mean(iou_diff):.4f}")
#     print(f"Paired t-test IoU:      t = {t_iou:.4f},  p = {p_iou:.4f}")
#     print(f"Wilcoxon test IoU:      W = {w_iou:.4f},  p = {p_w_iou:.4f}")
#     print()
